In [ ]:
import numpy as np
import torch
import torchvision
import torch.nn as nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import time

In [4]:
batch_size = 64
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
transform = torchvision.transforms.Compose([
    torchvision.transforms.CenterCrop((160, 160)),
    torchvision.transforms.Resize([64, 64]),
    torchvision.transforms.ToTensor(),
    torchvision.transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

train_data = datasets.CelebA(root='data',
                             split='train', transform=transform,
                             download=True)
test_data = datasets.CelebA(root='data',
                            split='test', transform=transform)

train_loader = DataLoader(dataset=train_data,
                              batch_size=batch_size,
                              num_workers=0,
                              shuffle=True)
test_loader = DataLoader(dataset=test_data,
                              batch_size=batch_size,
                              num_workers=0,
                              shuffle=False)

Downloading...
From (original): https://drive.google.com/uc?id=0B7EVK8r0v71pZjFTYXZWM3FlRnM
From (redirected): https://drive.usercontent.google.com/download?id=0B7EVK8r0v71pZjFTYXZWM3FlRnM&confirm=t&uuid=cc06a521-5860-480d-b85d-b23887a40555
To: c:\Users\nishi\code\Git\Deep learn\GAN\data\celeba\img_align_celeba.zip
100%|██████████| 1.44G/1.44G [05:02<00:00, 4.77MB/s]
Downloading...
From: https://drive.google.com/uc?id=0B7EVK8r0v71pblRyaVFSWGxPY0U
To: c:\Users\nishi\code\Git\Deep learn\GAN\data\celeba\list_attr_celeba.txt
100%|██████████| 26.7M/26.7M [00:06<00:00, 4.02MB/s]
Downloading...
From: https://drive.google.com/uc?id=1_ee_0u7vcNLOfNLegJRHmolfH5ICW-XS
To: c:\Users\nishi\code\Git\Deep learn\GAN\data\celeba\identity_CelebA.txt
100%|██████████| 3.42M/3.42M [00:01<00:00, 1.83MB/s]
Downloading...
From: https://drive.google.com/uc?id=0B7EVK8r0v71pbThiMVRxWXZ4dU0
To: c:\Users\nishi\code\Git\Deep learn\GAN\data\celeba\list_bbox_celeba.txt
100%|██████████| 6.08M/6.08M [00:02<00:00, 2.86MB

In [ ]:
print('Training Set:\n')
for images, labels in train_loader:  
    print('Image batch dimensions:', images.size())
    print('Image label dimensions:', labels.size())
    #print(labels[:10])
    break

In [ ]:
class GAN(torch.nn.modules):
    def __init__(self, latent_dim=100, 
                 num_feat_maps_gen=64, num_feat_maps_dis=64,
                 color_channels=3):
        super().__init__()
    
        self.fc = nn.Linear(latent_dim, num_feat_maps_gen*8*4*4)

        self.generator = nn.Sequential(
            nn.Upsample(scale_factor=2, mode='nearest'),  # or 'bilinear'
            nn.Conv2d(num_feat_maps_gen*8, num_feat_maps_gen*4, kernel_size=3, stride=1, padding=1),
            nn.ReLU(inplace=True),

            nn.Upsample(scale_factor=2, mode='nearest'),  # or 'bilinear'
            nn.Conv2d(num_feat_maps_gen*4, num_feat_maps_gen*4, kernel_size=3, stride=1, padding=1),
            nn.ReLU(inplace=True),

            nn.Upsample(scale_factor=2, mode='nearest'),  # or 'bilinear'
            nn.Conv2d(num_feat_maps_gen*4, num_feat_maps_gen*2, kernel_size=3, stride=1, padding=1),
            nn.ReLU(inplace=True),

            nn.Upsample(scale_factor=2, mode='nearest'),  # or 'bilinear'
            nn.Conv2d(num_feat_maps_gen*2, num_feat_maps_gen, kernel_size=3, stride=1, padding=1),
            nn.ReLU(inplace=True),

            nn.Upsample(scale_factor=2, mode='nearest'),  # or 'bilinear'
            nn.Conv2d(num_feat_maps_gen, color_channels, kernel_size=3, stride=1, padding=1),
            nn.ReLU(inplace=True),
        )

        self.discriminator = nn.Sequential(
            nn.Conv2d(color_channels, num_feat_maps_dis, kernel_size=3, stride=1, padding=1),
            nn.ReLU(inplace=True),

            nn.Conv2d(num_feat_maps_dis, num_feat_maps_dis*2, kernel_size=3, stride=1, padding=1),
            nn.ReLU(inplace=True),

            nn.Conv2d(num_feat_maps_dis*2, num_feat_maps_dis*4, kernel_size=3, stride=1, padding=1),
            nn.ReLU(inplace=True),

            nn.Conv2d(num_feat_maps_dis*4, num_feat_maps_dis*8, kernel_size=3, stride=1, padding=1),
            nn.ReLU(inplace=True),

            nn.Conv2d(num_feat_maps_dis*8, 1, kernel_size=3, stride=1, padding=1),
            nn.ReLU(inplace=True),

            nn.Flatten(),
        )

    def generator_forward(self, z):
        z = self.fc(z).view(z.size(0), -1, 4, 4)
        img = self.generator(z)
        return img
    
    def discriminator_forward(self, img):
        logits = self.discriminator(img)
        return logits

TypeError: module() takes at most 2 arguments (3 given)

In [ ]:
def train(model, device, train_loader, latent_dim, optimizer_gen, optimizer_dis, num_epochs):
    start_time = time.time()
    gen_minibatch_loss, dis_minibatch_loss = [],[]

    loss_fn = torch.nn.functional.binary_cross_entropy_with_logits

    for epoch in range(num_epochs):
        model.train()

        for batch_idx, (features,_) in enumerate(train_loader):

            real_images = features.to(device)
            real_labels = torch.ones(features.size(0), device=device)

            noice = torch.randn(features.size(0), latent_dim, 1, 1, device=device)
            fake_images = model.genrator_forward(noice)
            fake_labels = torch.zeros(features.size(0), device=device)
            fliped_fake_labels = real_labels

            optimizer_dis.zero_grad()

            pred_real = model.discriminator_forward(real_images).view(-1)
            real_loss = loss_fn(pred_real, real_labels)

            pred_fake = model.discriminator_forward(fake_images).view(-1)
            fake_loss = loss_fn(pred_fake, fake_labels)

            loss = (real_loss + fake_loss)*0.5
            loss.backward()

            optimizer_dis.step()


            optimizer_gen.zero_grad()
            

            